# FAP MultiVI subclustering

Recompute neighbors, UMAP, and Leiden clustering at resolutions 0.3-0.7 for `fap_multivi_orig_ident_complete.h5mu`, then draw one UMAP PDF and one marker dotplot PDF per resolution.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import mudata as mu
import anndata as ad
import scanpy as sc
from scipy import sparse

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)


In [ ]:
SOURCE_H5MU = Path('/Users/mingkewu/Documents/skeletal/mutivi/cornell+bch/fap_multivi_orig_ident_complete.h5mu')
OUT_DIR = SOURCE_H5MU.parent / 'fap_multivi_subcluster_res0.3_0.7'
OUT_DIR.mkdir(exist_ok=True)

RESOLUTIONS = [0.3, 0.4, 0.5, 0.6, 0.7]
FEATURE_GENES = ['MME', 'LUM', 'CD55', 'GPC3', 'COL11A1', 'ACTA1']


MARKER_GROUPS = {
    'MME+': ['MME', 'SMOC2', 'COL15A1', 'LAMA2'],
    'LUM+': ['LUM', 'CXCL14', 'PTGDS', 'PRG4'],
    'CD55+': ['CD55', 'MFAP5', 'DPP4'],
    'GPC3+': ['GPC3', 'CNTN4', 'NRP1', 'FBLN1'],
    'COL11A1+': ['COL11A1', 'COL11A2', 'THBS4', 'BMPR1B'],
    'ACTA1+': ['ACTA1', 'CKM', 'TTN', 'TNNC2'],
}

RNA_MARKER_GROUPS = MARKER_GROUPS


In [ ]:
def res_label(resolution):
    return f'{resolution:.1f}'


def cluster_sort_key(value):
    value = str(value)
    return (0, int(value)) if value.isdigit() else (1, value)


def clean_obs(df):
    clean = df.copy()
    clean.index = pd.Index(clean.index.astype(str), name=None)
    for col in clean.columns:
        values = clean[col]
        if isinstance(values.dtype, pd.CategoricalDtype):
            clean[col] = values.astype(str).astype('category')
        elif pd.api.types.is_object_dtype(values) or pd.api.types.is_string_dtype(values):
            clean[col] = values.astype(str)
    return clean


def marker_expression_adata(mdata, obs, marker_groups=MARKER_GROUPS):
    rna = mdata.mod['rna']
    var_names = set(map(str, rna.var_names))
    present_groups = {}
    missing = []

    for group, genes in marker_groups.items():
        present = [gene for gene in genes if gene in var_names]
        if present:
            present_groups[group] = present
        missing.extend([gene for gene in genes if gene not in var_names])

    genes = [gene for group_genes in present_groups.values() for gene in group_genes]
    X = rna[:, genes].X
    X = X.tocsr().astype(np.float32) if sparse.issparse(X) else sparse.csr_matrix(X, dtype=np.float32)

    if 'nCount_RNA' in obs.columns:
        totals = pd.to_numeric(obs['nCount_RNA'], errors='coerce').fillna(0).to_numpy(dtype=np.float64)
    else:
        totals = np.asarray(rna.X.sum(axis=1)).ravel().astype(np.float64)
    if np.count_nonzero(totals) == 0:
        totals = np.asarray(rna.X.sum(axis=1)).ravel().astype(np.float64)

    scale = np.divide(1e4, totals, out=np.zeros_like(totals), where=totals > 0)
    X = X.multiply(scale[:, None]).tocsr()
    X.data = np.log1p(X.data)

    marker_adata = ad.AnnData(X=X, obs=obs.copy(), var=pd.DataFrame(index=pd.Index(genes, name=None)))
    return marker_adata, present_groups, missing


In [ ]:
mdata = mu.read_h5mu(SOURCE_H5MU, backed='r')
obs = clean_obs(mdata.obs)

adata = ad.AnnData(X=np.asarray(mdata.obsm['X_multivi'], dtype=np.float32), obs=obs)
adata.obsm['X_multivi'] = adata.X.copy()

sc.pp.neighbors(adata, use_rep='X_multivi', n_neighbors=15, metric='euclidean', random_state=0)
sc.tl.umap(adata, min_dist=0.2, random_state=0)

cluster_counts = {}
for res in RESOLUTIONS:
    key = f'leiden_fap_multivi_res{res_label(res)}'
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        random_state=0,
        flavor='igraph',
        n_iterations=2,
        directed=False,
    )
    categories = sorted(adata.obs[key].astype(str).unique(), key=cluster_sort_key)
    adata.obs[key] = pd.Categorical(adata.obs[key].astype(str), categories=categories, ordered=True)
    cluster_counts[key] = adata.obs[key].value_counts().sort_index()

cluster_counts


In [ ]:
leiden_keys = [f'leiden_fap_multivi_res{res_label(res)}' for res in RESOLUTIONS]

for key in leiden_keys:
    sc.pl.umap(adata, color=key, legend_loc='on data', frameon=False, size=1.5, show=False)
    plt.savefig(OUT_DIR / f'umap_{key}.pdf', bbox_inches='tight')
    plt.close()

sorted(OUT_DIR.glob('umap_*.pdf'))


In [ ]:
marker_adata, present_marker_groups, missing_genes = marker_expression_adata(mdata, adata.obs, RNA_MARKER_GROUPS)
print('Missing marker genes:', ', '.join(missing_genes) if missing_genes else 'None')
present_marker_groups


In [ ]:
for key in leiden_keys:
    n_genes = sum(len(v) for v in present_marker_groups.values())
    n_clusters = marker_adata.obs[key].nunique()
    dp = sc.pl.dotplot(
        marker_adata,
        var_names=present_marker_groups,
        groupby=key,
        use_raw=False,
        standard_scale='var',
        figsize=(max(12, n_genes * 0.34), max(4, n_clusters * 0.28)),
        show=False,
        return_fig=True,
    )
    dp.savefig(OUT_DIR / f'dotplot_fap_markers_{key}.pdf')
    plt.close('all')

sorted(OUT_DIR.glob('dotplot_*.pdf'))


In [ ]:
feature_genes = [gene for gene in FEATURE_GENES if gene in marker_adata.var_names]
missing_feature_genes = [gene for gene in FEATURE_GENES if gene not in marker_adata.var_names]
print('Missing feature genes:', ', '.join(missing_feature_genes) if missing_feature_genes else 'None')

marker_adata.obsm['X_umap'] = adata.obsm['X_umap'].copy()

sc.pl.umap(
    marker_adata,
    color=feature_genes,
    frameon=False,
    size=1.5,
    ncols=3,
    cmap='viridis',
    vmax='p99',
    sort_order=True,
    show=False,
)
plt.savefig(OUT_DIR / 'featureplot_fap_markers_MME_LUM_CD55_GPC3_COL11A1_ACTA1.pdf', bbox_inches='tight')
plt.close('all')

for gene in feature_genes:
    sc.pl.umap(
        marker_adata,
        color=gene,
        frameon=False,
        size=1.5,
        cmap='viridis',
        vmax='p99',
        sort_order=True,
        show=False,
    )
    plt.savefig(OUT_DIR / f'featureplot_{gene}.pdf', bbox_inches='tight')
    plt.close('all')

sorted(OUT_DIR.glob('featureplot*.pdf'))


In [ ]:
from zipfile import ZipFile
import re
import xml.etree.ElementTree as ET
from scipy.stats import spearmanr
from matplotlib.ticker import PercentFormatter

META_XLSX = Path('/Users/mingkewu/Documents/skeletal/Ped_SkM_Sample_Meta.xlsx')
LEIDEN_RES07_KEY = 'leiden_fap_multivi_res0.7'
MERGED_RES07_KEY = 'fap_multivi_annotation_res0.7'
AGE_KEY = 'age_num'
AGE_GROUP_KEY = 'age_group'
SAMPLE_KEY = 'orig.ident'

RES07_CLUSTER_TO_SUBTYPE = {
    '0': 'CD55+',
    '2': 'MME+', '3': 'MME+', '4': 'MME+', '5': 'MME+',
    '7': 'LUM+',
    '1': 'GPC3+', '8': 'GPC3+', '10': 'GPC3+', '11': 'GPC3+',
    '9': 'COL11A1+',
    '6': 'ACTA1+',
    '12': 'Unknown',
}
MERGED_SUBTYPE_ORDER = ['MME+', 'LUM+', 'CD55+', 'GPC3+', 'COL11A1+', 'ACTA1+', 'Unknown']
AGE_SUBTYPE_ORDER = ['MME+', 'LUM+', 'CD55+', 'GPC3+', 'COL11A1+', 'ACTA1+']
MERGED_PALETTE = {
    'MME+': '#4C78A8', 'LUM+': '#F58518', 'CD55+': '#54A24B',
    'GPC3+': '#B279A2', 'COL11A1+': '#E45756', 'ACTA1+': '#72B7B2',
    'Unknown': '#999999',
}
AGE_GROUP_ORDER = ['Infant', 'Early Childhood', 'Late Childhood', 'Adolescence']
AGE_GROUP_COLORS = {
    'Infant': '#6BAED6', 'Early Childhood': '#74C476',
    'Late Childhood': '#FDAE6B', 'Adolescence': '#9E9AC8',
}

missing = sorted(set(adata.obs[LEIDEN_RES07_KEY].astype(str)) - set(RES07_CLUSTER_TO_SUBTYPE), key=cluster_sort_key)
if missing:
    raise ValueError(f'Missing subtype annotation for clusters: {missing}')

adata.obs[MERGED_RES07_KEY] = pd.Categorical(
    adata.obs[LEIDEN_RES07_KEY].astype(str).map(RES07_CLUSTER_TO_SUBTYPE),
    categories=MERGED_SUBTYPE_ORDER,
    ordered=True,
)
marker_adata.obs[MERGED_RES07_KEY] = adata.obs[MERGED_RES07_KEY].copy()

sc.pl.umap(
    adata,
    color=MERGED_RES07_KEY,
    palette=MERGED_PALETTE,
    legend_loc='on data',
    frameon=False,
    size=1.5,
    show=False,
)
plt.savefig(OUT_DIR / 'umap_fap_merged_annotation_res0.7.pdf', bbox_inches='tight')
plt.close('all')

n_genes = sum(len(v) for v in present_marker_groups.values())
dp = sc.pl.dotplot(
    marker_adata,
    var_names=present_marker_groups,
    groupby=MERGED_RES07_KEY,
    categories_order=MERGED_SUBTYPE_ORDER,
    use_raw=False,
    standard_scale='var',
    figsize=(max(12, n_genes * 0.34), 4.0),
    show=False,
    return_fig=True,
)
dp.savefig(OUT_DIR / 'dotplot_fap_merged_annotation_res0.7.pdf')
plt.close('all')

NS = {'a': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}

def _xlsx_col_to_idx(ref):
    letters = ''.join(ch for ch in ref if ch.isalpha())
    idx = 0
    for ch in letters:
        idx = idx * 26 + ord(ch.upper()) - 64
    return idx - 1


def read_xlsx_first_sheet(path):
    with ZipFile(path) as z:
        shared = []
        if 'xl/sharedStrings.xml' in z.namelist():
            root = ET.fromstring(z.read('xl/sharedStrings.xml'))
            for si in root.findall('a:si', NS):
                shared.append(''.join(t.text or '' for t in si.findall('.//a:t', NS)))
        sheet = ET.fromstring(z.read('xl/worksheets/sheet1.xml'))
        rows = []
        for row in sheet.findall('.//a:sheetData/a:row', NS):
            values = {}
            for cell in row.findall('a:c', NS):
                idx = _xlsx_col_to_idx(cell.attrib.get('r', 'A'))
                typ = cell.attrib.get('t')
                node = cell.find('a:v', NS)
                val = '' if node is None else (node.text or '')
                if typ == 's' and val:
                    val = shared[int(val)]
                elif typ == 'inlineStr':
                    val = ''.join(t.text or '' for t in cell.findall('.//a:t', NS))
                values[idx] = val
            if values:
                rows.append([values.get(i, '') for i in range(max(values) + 1)])
    width = max(len(row) for row in rows)
    rows = [row + [''] * (width - len(row)) for row in rows]
    return pd.DataFrame(rows[1:], columns=rows[0])


def parse_age_years(value):
    text = str(value).strip().lower()
    match = re.search(r'([0-9]+(?:\\.[0-9]+)?)', text)
    if not match:
        return np.nan
    age = float(match.group(1))
    return age / 12.0 if 'mo' in text or 'month' in text else age


def donor_candidates(orig_ident):
    base = str(orig_ident).split('_')[0]
    candidates = [base]
    if base.startswith('CTRL'):
        stripped = re.sub(r'([a-z]+)$', '', base)
        if stripped != base:
            candidates.append(stripped)
    if base.isdigit() and len(base) > 6:
        candidates.extend([base[:-1], base[:6]])
    return list(dict.fromkeys(candidates))


def bh_fdr(pvalues):
    pvalues = np.asarray(pvalues, dtype=float)
    out = np.full_like(pvalues, np.nan, dtype=float)
    valid = np.isfinite(pvalues)
    vals = pvalues[valid]
    if vals.size == 0:
        return out
    order = np.argsort(vals)
    ranks = np.arange(1, vals.size + 1)
    q = vals[order] * vals.size / ranks
    q = np.minimum.accumulate(q[::-1])[::-1]
    tmp = np.empty_like(vals)
    tmp[order] = np.clip(q, 0, 1)
    out[valid] = tmp
    return out


def format_p(value):
    if not np.isfinite(value):
        return 'NA'
    return f'{value:.1e}' if value < 0.001 else f'{value:.3f}'


meta = read_xlsx_first_sheet(META_XLSX)
meta['donor_id'] = meta['Donor ID'].astype(str)
meta[AGE_KEY] = meta['Age'].map(parse_age_years)
meta[AGE_GROUP_KEY] = pd.cut(meta[AGE_KEY], [-np.inf, 3, 6, 13, np.inf], right=False, labels=AGE_GROUP_ORDER)
age_lookup = meta.set_index('donor_id')[[AGE_KEY, AGE_GROUP_KEY]].to_dict()

donors, ages, age_groups = [], [], []
for sample in adata.obs[SAMPLE_KEY].astype(str):
    donor = next((cand for cand in donor_candidates(sample) if cand in age_lookup[AGE_KEY]), None)
    donors.append(donor)
    ages.append(age_lookup[AGE_KEY].get(donor, np.nan))
    age_groups.append(age_lookup[AGE_GROUP_KEY].get(donor, np.nan))

adata.obs['donor_id_for_age'] = donors
adata.obs[AGE_KEY] = ages
adata.obs[AGE_GROUP_KEY] = pd.Categorical(age_groups, categories=AGE_GROUP_ORDER, ordered=True)

df = adata.obs[[SAMPLE_KEY, MERGED_RES07_KEY, AGE_KEY, AGE_GROUP_KEY]].dropna(subset=[SAMPLE_KEY, MERGED_RES07_KEY, AGE_KEY]).copy()
counts = df.groupby([SAMPLE_KEY, MERGED_RES07_KEY], observed=True).size().rename('n').reset_index()
totals = df.groupby(SAMPLE_KEY, observed=True).size().rename('total_cells').reset_index()
counts = counts.merge(totals, on=SAMPLE_KEY, how='left')
counts['fraction'] = counts['n'] / counts['total_cells']
grid = pd.MultiIndex.from_product([df[SAMPLE_KEY].dropna().unique(), AGE_SUBTYPE_ORDER], names=[SAMPLE_KEY, MERGED_RES07_KEY])
fractions = counts.set_index([SAMPLE_KEY, MERGED_RES07_KEY]).reindex(grid, fill_value=0).reset_index()
sample_meta = df[[SAMPLE_KEY, AGE_KEY, AGE_GROUP_KEY]].drop_duplicates(SAMPLE_KEY)
fractions = fractions.drop(columns=[AGE_KEY, AGE_GROUP_KEY], errors='ignore').merge(sample_meta, on=SAMPLE_KEY, how='left')
fractions = fractions.rename(columns={SAMPLE_KEY: 'sample', MERGED_RES07_KEY: 'subtype'})
fractions['subtype'] = pd.Categorical(fractions['subtype'], categories=AGE_SUBTYPE_ORDER, ordered=True)

stats_rows = []
for subtype in AGE_SUBTYPE_ORDER:
    sub = fractions[fractions['subtype'].astype(str) == subtype].dropna(subset=[AGE_KEY, 'fraction'])
    if sub[AGE_KEY].nunique() < 3 or sub['fraction'].nunique() < 2:
        rho, pval = np.nan, np.nan
    else:
        rho, pval = spearmanr(sub[AGE_KEY], sub['fraction'])
    stats_rows.append({'subtype': subtype, 'rho': rho, 'pval': pval, 'n_samples': len(sub)})
stats = pd.DataFrame(stats_rows)
stats['qval'] = bh_fdr(stats['pval'].values)

fig, axes = plt.subplots(2, 3, figsize=(10, 5.8), sharey=False)
for ax, subtype in zip(axes.ravel(), AGE_SUBTYPE_ORDER):
    sub = fractions[fractions['subtype'].astype(str) == subtype].dropna(subset=[AGE_KEY, 'fraction'])
    for group in AGE_GROUP_ORDER:
        ss = sub[sub[AGE_GROUP_KEY].astype(str) == group]
        if len(ss):
            ax.scatter(ss[AGE_KEY], ss['fraction'], s=22, color=AGE_GROUP_COLORS[group], alpha=0.8, linewidth=0, label=group)
    if len(sub) >= 3 and sub[AGE_KEY].nunique() > 1:
        coef = np.polyfit(sub[AGE_KEY], sub['fraction'], deg=1)
        xs = np.linspace(sub[AGE_KEY].min(), sub[AGE_KEY].max(), 100)
        ax.plot(xs, coef[0] * xs + coef[1], color='black', lw=1)
    row = stats[stats['subtype'] == subtype].iloc[0]
    ax.text(0.03, 0.95, f"rho={row['rho']:.2f}\np={format_p(row['pval'])}, FDR={format_p(row['qval'])}", transform=ax.transAxes, va='top', ha='left', fontsize=8)
    ax.set_title(subtype, fontsize=10, loc='left')
    ax.set_xlabel('Age')
    ax.set_ylabel('Sample-level cell fraction')
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.grid(axis='y', color='0.86', linewidth=0.6)
    ax.spines[['top', 'right']].set_visible(False)

handles, labels = axes.ravel()[0].get_legend_handles_labels()
axes.ravel()[0].legend(handles, labels, loc='lower left', frameon=False, fontsize=7)
fig.tight_layout()
fig.savefig(OUT_DIR / 'number_continuous_age_fap_merged_annotation_res0.7.pdf', bbox_inches='tight')
plt.close(fig)

adata.obs[[SAMPLE_KEY, 'donor_id_for_age', AGE_KEY, AGE_GROUP_KEY, LEIDEN_RES07_KEY, MERGED_RES07_KEY]].to_csv(OUT_DIR / 'fap_merged_annotation_res0.7_counts.csv')
fractions.to_csv(OUT_DIR / 'fap_merged_annotation_res0.7_sample_fractions.csv', index=False)
stats.to_csv(OUT_DIR / 'fap_merged_annotation_res0.7_age_spearman.csv', index=False)

adata.obs[MERGED_RES07_KEY].value_counts().reindex(MERGED_SUBTYPE_ORDER)


In [ ]:
import gzip
import snapatac2 as snap
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

GENCODE_GTF = Path('/tmp/gencode.v50.annotation.gtf.gz')
MARKER_GTF = Path('/tmp/gencode.v50.fap_marker_genes_classic.gtf.gz')
GENE_NAME_RE = re.compile(r'gene_name "([^"]+)"')

GA_DOTPLOT_PDF = OUT_DIR / 'gene_activity_dotplot_fap_merged_annotation_res0.7_cornell_atac.pdf'

marker_genes = []
for genes in MARKER_GROUPS.values():
    for gene in genes:
        if gene not in marker_genes:
            marker_genes.append(gene)

with gzip.open(GENCODE_GTF, 'rt') as source, gzip.open(MARKER_GTF, 'wt') as target:
    found = set()
    for line in source:
        if line.startswith('#'):
            continue
        match = GENE_NAME_RE.search(line)
        if match and match.group(1) in marker_genes:
            target.write(line)
            found.add(match.group(1))
print('Marker genes missing from GENCODE:', sorted(set(marker_genes) - found))

has_atac_values = adata.obs['has_atac']
if has_atac_values.dtype == bool:
    has_atac = has_atac_values.to_numpy()
else:
    has_atac = has_atac_values.astype(str).str.lower().isin(['true', '1', 'yes']).to_numpy()
atac_indices = np.flatnonzero(has_atac)
atac_obs = adata.obs.iloc[atac_indices].copy()
atac_obs[MERGED_RES07_KEY] = pd.Categorical(atac_obs[MERGED_RES07_KEY].astype(str), categories=MERGED_SUBTYPE_ORDER, ordered=True)
print(atac_obs[MERGED_RES07_KEY].value_counts().reindex(MERGED_SUBTYPE_ORDER))

atac = mdata.mod['atac'][atac_indices, :].to_memory()
atac.obs = clean_obs(atac_obs)
atac.var = pd.DataFrame(index=atac.var_names.astype(str))
if sparse.issparse(atac.X):
    atac.X = atac.X.tocsr().astype(np.uint32)
else:
    atac.X = np.asarray(atac.X, dtype=np.uint32)

gene_activity = snap.pp.make_gene_matrix(
    atac,
    gene_anno=MARKER_GTF,
    use_x=True,
    inplace=False,
    upstream=2000,
    downstream=0,
    include_gene_body=True,
    chunk_size=500,
)
gene_activity.X = gene_activity.X.tocsr() if sparse.issparse(gene_activity.X) else sparse.csr_matrix(gene_activity.X)

totals = pd.to_numeric(atac.obs['nCount_ATAC'], errors='coerce').fillna(0).to_numpy(dtype=np.float64)
scale = np.divide(1e4, totals, out=np.zeros_like(totals), where=totals > 0)
gene_activity.X = gene_activity.X.multiply(scale[:, None]).tocsr()
gene_activity.X.data = np.log1p(gene_activity.X.data)
gene_activity.obs = atac.obs[[MERGED_RES07_KEY, LEIDEN_RES07_KEY, 'orig.ident', 'cohort', 'has_atac']].copy()
gene_activity.obs[MERGED_RES07_KEY] = pd.Categorical(gene_activity.obs[MERGED_RES07_KEY].astype(str), categories=MERGED_SUBTYPE_ORDER, ordered=True)

available = set(gene_activity.var_names.astype(str))
nonzero = set(gene_activity.var_names[np.asarray(gene_activity.X.sum(axis=0)).ravel() > 0].astype(str))
ga_marker_groups = {}
gene_status = []
for group, genes in MARKER_GROUPS.items():
    present = []
    for gene in genes:
        status = 'present_nonzero' if gene in available and gene in nonzero else ('present_zero' if gene in available else 'absent')
        gene_status.append({'group': group, 'gene': gene, 'status': status})
        if status == 'present_nonzero':
            present.append(gene)
    if present:
        ga_marker_groups[group] = present
print(pd.DataFrame(gene_status).query("status != 'present_nonzero'"))

def dot_size(pct):
    return 6 + np.power(np.clip(pct, 0, 100) / 100.0, 1.7) * 900


def make_enhanced_gene_activity_dotplot(gene_activity, marker_groups, out_pdf):
    genes = [gene for group_genes in marker_groups.values() for gene in group_genes]
    categories = [x for x in MERGED_SUBTYPE_ORDER if x in set(gene_activity.obs[MERGED_RES07_KEY].astype(str))]
    X = gene_activity[:, genes].X
    X = X.tocsr() if sparse.issparse(X) else sparse.csr_matrix(X)
    labels = gene_activity.obs[MERGED_RES07_KEY].astype(str).to_numpy()

    mean = np.zeros((len(categories), len(genes)), dtype=float)
    pct = np.zeros_like(mean)
    for i, category in enumerate(categories):
        idx = np.flatnonzero(labels == category)
        Xg = X[idx]
        mean[i] = np.asarray(Xg.mean(axis=0)).ravel()
        pct[i] = np.asarray((Xg > 0).mean(axis=0)).ravel() * 100

    scaled = mean.copy()
    for j in range(scaled.shape[1]):
        lo, hi = np.nanmin(scaled[:, j]), np.nanmax(scaled[:, j])
        scaled[:, j] = 0 if hi <= lo else (scaled[:, j] - lo) / (hi - lo)
    color_values = np.power(np.clip(scaled, 0, 1), 0.75)

    fig, ax = plt.subplots(figsize=(max(16.5, len(genes) * 0.64), max(5.3, len(categories) * 0.52 + 1.7)))
    xs, ys = np.meshgrid(np.arange(len(genes)), np.arange(len(categories)))
    ax.scatter(
        xs.ravel(), ys.ravel(),
        s=dot_size(pct).ravel(),
        c=color_values.ravel(),
        cmap='YlOrRd',
        vmin=0,
        vmax=1,
        edgecolors='0.25',
        linewidths=0.35,
    )
    ax.set_xticks(np.arange(len(genes)))
    ax.set_xticklabels(genes, rotation=90, ha='center', va='top', fontsize=9)
    ax.set_yticks(np.arange(len(categories)))
    ax.set_yticklabels(categories, fontsize=10)
    ax.set_xlim(-0.8, len(genes) - 0.2)
    ax.set_ylim(len(categories) - 0.35, -1.55)
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.grid(axis='both', color='0.90', linewidth=0.6)
    ax.set_axisbelow(True)
    ax.spines[['top', 'right']].set_visible(False)

    start = 0
    for group, group_genes in marker_groups.items():
        present = [gene for gene in group_genes if gene in genes]
        if not present:
            continue
        end = start + len(present) - 1
        ax.plot([start - 0.45, end + 0.45], [-0.72, -0.72], color='black', lw=1.4, clip_on=False)
        ax.plot([start - 0.45, start - 0.45], [-0.72, -0.55], color='black', lw=1.4, clip_on=False)
        ax.plot([end + 0.45, end + 0.45], [-0.72, -0.55], color='black', lw=1.4, clip_on=False)
        ax.text((start + end) / 2, -1.25, group, ha='center', va='bottom', rotation=90, fontsize=11, clip_on=False)
        if end + 0.5 < len(genes) - 0.5:
            ax.axvline(end + 0.5, color='0.72', lw=0.8)
        start = end + 1

    cbar = fig.colorbar(ScalarMappable(norm=Normalize(0, 1), cmap='YlOrRd'), ax=ax, fraction=0.024, pad=0.03)
    cbar.set_label('Scaled mean gene activity\nwithin each gene', fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    for value in [10, 30, 50, 70, 90]:
        ax.scatter([], [], s=dot_size(value), c='white', edgecolors='0.25', linewidths=0.35, label=f'{value}%')
    legend = ax.legend(
        title='Cells with activity',
        loc='center left',
        bbox_to_anchor=(1.08, 0.48),
        frameon=False,
        fontsize=8,
        title_fontsize=9,
        scatterpoints=1,
        labelspacing=1.2,
    )
    for handle in legend.legend_handles:
        handle.set_facecolor('white')

    fig.tight_layout()
    fig.savefig(out_pdf, bbox_inches='tight')
    plt.close(fig)


make_enhanced_gene_activity_dotplot(gene_activity, ga_marker_groups, GA_DOTPLOT_PDF)
GA_DOTPLOT_PDF


In [ ]:
for key, counts in cluster_counts.items():
    print(key, 'n_clusters=', len(counts))
    print(counts.to_dict())

print('Output directory:', OUT_DIR)
for path in sorted(OUT_DIR.glob('*.pdf')):
    print(path.name)
mdata.file.close()
